In [1]:
import pandas as pd

customers = pd.read_csv("customers.csv")

customers = customers[
    ["customer_id", "customer_name"]
]

customers = customers.sample(40)

customers.to_csv("customers_clean.csv", index=False)


In [2]:
trucks = pd.read_csv("trucks.csv")

trucks = trucks[
    ["truck_id", "make", "vin"]
]

trucks = trucks.sample(40)

trucks.to_csv("trucks_clean.csv", index=False)

In [3]:
loads = pd.read_csv("loads.csv")

loads = loads[
    ["load_id", "customer_id", "route_id", "load_date", "load_status"]
]


loads.to_csv("loads_2.csv", index=False)

In [5]:
valid_customers = customers["customer_id"]
valid_trucks = trucks["truck_id"]



In [10]:
loads_2 = pd.read_csv("loads_2.csv")

loads_2["load_date"] = pd.to_datetime(loads_2["load_date"])

loads_clean = loads_2[
    (loads_2["load_date"].dt.year == 2024) &
    (loads_2["load_date"].dt.month.isin([11, 12])) &
    (loads_2["customer_id"].isin(valid_customers))
]

loads_clean.to_csv("loads_clean.csv", index=False)

In [15]:
trips = pd.read_csv("trips.csv")

trips = trips[
    ["trip_id", "load_id", "truck_id", "dispatch_date", "actual_distance_miles", "trip_status"]
]

valid_loads = loads_clean["load_id"]

trips_clean = trips[
    (trips["load_id"].isin(valid_loads)) &
    (trips["truck_id"].isin(valid_trucks))
]

trips_clean.to_csv("trips_clean.csv", index=False)

In [20]:
fuel_purchases = pd.read_csv("fuel_purchases.csv")

fuel_purchases = fuel_purchases[
    ["fuel_purchase_id", "trip_id", "truck_id", "purchase_date", "total_cost"]
]

valid_trips = trips_clean["trip_id"]

fuel_purchases_clean = fuel_purchases[
    fuel_purchases["trip_id"].isin(valid_trips)
]

fuel_purchases_clean.to_csv("fuel_purchases_clean.csv", index=False)

In [17]:
maintenance = pd.read_csv("maintenance_records.csv")

maintenance = maintenance[
    ["maintenance_id", "truck_id", "maintenance_date", "total_cost"]
]

maintenance["maintenance_date"] = pd.to_datetime(maintenance["maintenance_date"])

maintenance_clean = maintenance[ 
    (maintenance["truck_id"].isin(valid_trucks)) &
    (maintenance["maintenance_date"].dt.year == 2024) &
    (maintenance["maintenance_date"].dt.month.isin([11, 12]))

]

maintenance_clean.to_csv("maintenance_clean.csv", index=False)

In [18]:
events = pd.read_csv("delivery_events.csv")
events = events[  
    ["event_id", "load_id", "trip_id", "event_type", "actual_datetime", "location_city"]
]

events_clean = events[  
    events["trip_id"].isin(valid_trips)
 ]

events_clean.to_csv("events_clean.csv", index=False)

In [19]:
routes = pd.read_csv("routes.csv")

routes = routes[ 
    ["route_id", "origin_city", "destination_city"]
]

valid_routes = loads_clean["route_id"]

routes_clean = routes[ 
    routes["route_id"].isin(valid_routes)
]

routes_clean.to_csv("routes_clean.csv")



In [22]:
fuel = fuel_purchases_clean.rename(columns={
    "fuel_purchase_id": "expense_id",
    "trip_id": "delivery_id",
    "purchase_date": "expense_date",
    "total_cost": "amount"
})

fuel["expense_type"] = "Fuel"

fuel = fuel[
    [
        "expense_id",
        "delivery_id",
        "expense_type",
        "expense_date",
        "amount"
    ]
]

In [23]:
maintenance_0 = maintenance_clean.rename(columns={
    "maintenance_id": "expense_id",
    "truck_id": "delivery_id",   # temporary compromise
    "total_cost": "amount",
    "maintenance_date": "expense_date"
})

maintenance_0["expense_type"] = "Maintenance"

maintenance_0 = maintenance_0[
    [
        "expense_id",
        "delivery_id",
        "expense_type",
        "expense_date",
        "amount"
    ]
]



In [26]:
expenses = pd.concat([fuel, maintenance_0], ignore_index=True)

expenses.to_csv("expenses.csv", index=False)